In [ ]:
import os,sys,json,hashlib,zipfile,io,subprocess,shutil,time
SESSION_STARTED=time.monotonic()
from pathlib import Path
from datetime import datetime,timezone
RUN_DIR=Path('/kaggle/working/s1_infra_recovery')/'r03'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True,exist_ok=False)
SOURCE=RUN_DIR/'source'
payload=__import__('base64').b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='682d1782daceb2f34fb7007d1ca0cb94f287fc7169c8020d47ddd50bf3572b27'
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert z.testzip() is None
    assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(SOURCE)
sys.path.insert(0,str(SOURCE))
REG=json.loads((SOURCE/'s1/registration.json').read_text())
for n,h in REG['source_hashes'].items():assert hashlib.sha256((SOURCE/n).read_bytes()).hexdigest()==h,n
WHEELHOUSE_DIR=next(Path('/kaggle/input').rglob('swegemma-0.2.11-py3-none-any.whl')).parent
actual={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in WHEELHOUSE_DIR.glob('*.whl') if 'cutlass' not in p.name.lower()}
assert actual==REG['wheel_hashes'],'Official wheel drift'
(RUN_DIR/'wheel_hashes.json').write_text(json.dumps(actual,indent=2))


In [ ]:
from harness_v2.evidence import bundle
try:
    import glob
    os.environ.update(TRANSFORMERS_NO_TF='1',TRANSFORMERS_NO_FLAX='1',TRANSFORMERS_OFFLINE='1',HF_HUB_OFFLINE='1',LITELLM_LOCAL_MODEL_COST_MAP='True',LITELLM_TELEMETRY='False',OTEL_SDK_DISABLED='true',PYTHONIOENCODING='utf-8')
    for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth','/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for path in glob.glob(pattern):Path(path).unlink(missing_ok=True)
    links=RUN_DIR/'wheelhouse_links';links.mkdir()
    try:
        for p in WHEELHOUSE_DIR.glob('*.whl'):
            if 'cutlass' in p.name.lower():continue
            name=p.name.replace('cu128','+cu128') if 'cu128' in p.name and '+' not in p.name else p.name
            (links/name).symlink_to(p)
        with (RUN_DIR/'installation.log').open('x') as log:
            subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--force-reinstall']+sorted(str(p) for p in links.glob('*.whl')),stdout=log,stderr=subprocess.STDOUT,check=True,timeout=1800)
    finally:shutil.rmtree(links)
    
    p=subprocess.run([sys.executable,'-m','s1.infrastructure_recovery.cpu',str(RUN_DIR/'validation')],cwd=SOURCE,capture_output=True,text=True,timeout=1800)
    (RUN_DIR/'runner.stdout.log').write_text(p.stdout);(RUN_DIR/'runner.stderr.log').write_text(p.stderr)
    print(p.stdout+p.stderr,flush=True)
    assert p.returncode==0,'CPU investigation/gate failed'
finally:print('INFRA_CPU_EVIDENCE',bundle(RUN_DIR),flush=True)
